# 卷积基础与原理

本notebook介绍卷积神经网络(CNN)的基本原理:
- 为什么需要卷积神经网络
- 卷积运算的数学原理
- 互相关运算
- 填充(padding)和步幅(stride)
- 多通道卷积
- 池化(pooling)层

CNN是计算机视觉的基础!

## 第一部分: 为什么需要卷积神经网络

### 1.1 全连接层的问题

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np

**问题1: 参数过多**

假设输入图像大小为 $1000 \times 1000$ (百万像素),第一层隐藏层有1000个神经元:
- 参数量: $10^6 \times 10^3 = 10^9$ (10亿个参数!)
- 存储: 约4GB内存(float32)
- 训练: 需要海量数据,容易过拟合

即使降低到 $100 \times 100$ 像素,仍需 $10^4 \times 10^3 = 10^7$ (千万级)参数。

In [ ]:
# 演示全连接层的参数量
image_sizes = [28, 100, 224, 500, 1000]
hidden_units = 1000

print("全连接层参数量分析:")
print("-" * 60)
for size in image_sizes:
    params = (size * size) * hidden_units
    memory_mb = params * 4 / (1024 ** 2)  # float32 占4字节
    print(f"图像大小: {size:4}×{size:4}, 参数量: {params:>12,}, 内存: {memory_mb:>7.1f} MB")

print("\n可视化")
params_millions = [(size * size * hidden_units) / 1e6 for size in image_sizes]

plt.figure(figsize=(10, 6))
plt.bar(range(len(image_sizes)), params_millions, color='steelblue', alpha=0.7)
plt.xlabel('图像大小', fontsize=12)
plt.ylabel('参数量(百万)', fontsize=12)
plt.title('全连接层参数量随图像大小增长', fontsize=14)
plt.xticks(range(len(image_sizes)), [f'{s}×{s}' for s in image_sizes])
plt.grid(axis='y', alpha=0.3)
for i, v in enumerate(params_millions):
    plt.text(i, v + 10, f'{v:.0f}M', ha='center', fontsize=10)
plt.tight_layout()
plt.show()

print("\n结论: 全连接层对高分辨率图像是灾难!")

**问题2: 忽略空间结构**

全连接层将图像展平为一维向量,丢失了:
- 像素的空间位置信息
- 邻近像素的相关性
- 图像的局部模式

但人类识别图像时,主要依赖局部特征(边缘、纹理、形状等)!

### 1.2 图像的两个重要特性

**特性1: 平移不变性 (Translation Invariance)**

无论物体出现在图像的哪个位置,我们都应该能识别它。

例如: 猫在左上角和右下角应该被识别为同一类物体。

**特性2: 局部性 (Locality)**

图像的邻近像素通常高度相关,而距离较远的像素相关性较弱。

例如: 判断一个像素是否是边缘,只需要看它周围的像素。

### 1.3 卷积神经网络的解决方案

CNN通过以下方式解决全连接层的问题:

1. **参数共享**: 同一个卷积核在整个图像上滑动,大幅减少参数
2. **局部连接**: 每个神经元只与局部区域连接,而不是全连接
3. **空间结构**: 保留图像的2D空间结构

**参数量对比**:
- 全连接: $1000 \times 1000 \times 1000 = 10^9$ 参数
- 卷积层: $3 \times 3 \times 64 = 576$ 参数 (64个3×3卷积核)
- 减少: **约100万倍**!

---

## 第二部分: 卷积运算

### 2.1 二维互相关运算

卷积层的核心是**互相关运算**(cross-correlation)。

给定输入 $\mathbf{X}$ 和卷积核 $\mathbf{K}$,输出:

$$[\mathbf{Y}]_{i,j} = \sum_a \sum_b [\mathbf{K}]_{a,b} \cdot [\mathbf{X}]_{i+a, j+b}$$

In [ ]:
def corr2d(X, K):
    """计算二维互相关运算"""
    h, w = K.shape
    Y = torch.zeros((X.shape[0] - h + 1, X.shape[1] - w + 1))
    for i in range(Y.shape[0]):
        for j in range(Y.shape[1]):
            Y[i, j] = (X[i:i + h, j:j + w] * K).sum()
    return Y

# 示例: 3×3输入, 2×2卷积核
X = torch.tensor([[0.0, 1.0, 2.0],
                   [3.0, 4.0, 5.0],
                   [6.0, 7.0, 8.0]])

K = torch.tensor([[0.0, 1.0],
                   [2.0, 3.0]])

print("输入 X:")
print(X)
print("\n卷积核 K:")
print(K)
print("\n输出 Y = X ⊗ K:")
Y = corr2d(X, K)
print(Y)

print("\n计算过程:")
print("Y[0,0] = 0×0 + 1×1 + 3×2 + 4×3 =", (0*0 + 1*1 + 3*2 + 4*3).item())
print("Y[0,1] = 1×0 + 2×1 + 4×2 + 5×3 =", (1*0 + 2*1 + 4*2 + 5*3).item())
print("Y[1,0] = 3×0 + 4×1 + 6×2 + 7×3 =", (3*0 + 4*1 + 6*2 + 7*3).item())
print("Y[1,1] = 4×0 + 5×1 + 7×2 + 8×3 =", (4*0 + 5*1 + 7*2 + 8*3).item())

### 2.2 可视化卷积运算

In [ ]:
def visualize_convolution(X, K):
    """可视化卷积过程"""
    h, w = K.shape
    out_h, out_w = X.shape[0] - h + 1, X.shape[1] - w + 1
    
    fig, axes = plt.subplots(2, out_w, figsize=(12, 6))
    
    for i in range(out_h):
        for j in range(out_w):
            # 显示输入区域
            ax = axes[i, j]
            window = X[i:i+h, j:j+w].numpy()
            ax.imshow(window, cmap='viridis', vmin=0, vmax=8)
            ax.set_title(f'位置({i},{j})', fontsize=10)
            ax.axis('off')
            
            # 添加数值
            for ii in range(h):
                for jj in range(w):
                    ax.text(jj, ii, f'{window[ii,jj]:.0f}', 
                           ha='center', va='center', color='white', fontsize=14)
    
    plt.suptitle('卷积窗口滑动过程', fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()

visualize_convolution(X, K)

### 2.3 边缘检测示例

卷积的一个经典应用: **检测图像边缘**

In [ ]:
# 创建一个简单的图像: 中间4列为黑色(0),其余为白色(1)
X = torch.ones((6, 8))
X[:, 2:6] = 0

print("输入图像:")
print(X)

# 垂直边缘检测核
K = torch.tensor([[1.0, -1.0]])

print("\n边缘检测卷积核:")
print(K)

Y = corr2d(X, K)
print("\n检测结果:")
print(Y)

# 可视化
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].imshow(X, cmap='gray')
axes[0].set_title('原始图像', fontsize=12)
axes[0].axis('off')

axes[1].imshow(K, cmap='coolwarm', vmin=-1, vmax=1)
axes[1].set_title('边缘检测核 [1, -1]', fontsize=12)
axes[1].axis('off')

axes[2].imshow(Y, cmap='coolwarm')
axes[2].set_title('边缘检测结果', fontsize=12)
axes[2].axis('off')

plt.tight_layout()
plt.show()

print("\n观察:")
print("- 在从白到黑的边缘处,输出为1")
print("- 在从黑到白的边缘处,输出为-1")
print("- 在均匀区域,输出为0")

### 2.4 学习卷积核

在实际应用中,卷积核的权重是通过训练学习得到的!

In [ ]:
# 定义简单的卷积层
class Conv2D(nn.Module):
    def __init__(self, kernel_size):
        super().__init__()
        self.weight = nn.Parameter(torch.rand(kernel_size))
        self.bias = nn.Parameter(torch.zeros(1))
    
    def forward(self, x):
        return corr2d(x, self.weight) + self.bias

# 训练学习边缘检测核
conv2d = Conv2D(kernel_size=(1, 2))

# 构造训练数据
X = torch.ones((6, 8))
X[:, 2:6] = 0
Y = corr2d(X, torch.tensor([[1.0, -1.0]]))

# 训练
lr = 3e-2
num_epochs = 10

print("训练前的卷积核:")
print(conv2d.weight.data)

for epoch in range(num_epochs):
    Y_hat = conv2d(X)
    loss = (Y_hat - Y) ** 2
    conv2d.zero_grad()
    loss.sum().backward()
    
    # 手动更新参数
    conv2d.weight.data -= lr * conv2d.weight.grad
    conv2d.bias.data -= lr * conv2d.bias.grad
    
    if (epoch + 1) % 2 == 0:
        print(f'Epoch {epoch+1}, Loss: {loss.sum():.4f}')

print("\n训练后的卷积核:")
print(conv2d.weight.data)
print("\n目标卷积核: [[1.0, -1.0]]")
print("学习成功!")

---

## 第三部分: 填充和步幅

### 3.1 输出大小计算

输入大小: $n_h \times n_w$  
卷积核大小: $k_h \times k_w$  
输出大小: $(n_h - k_h + 1) \times (n_w - k_w + 1)$

**问题**: 多层卷积后,特征图会越来越小!

In [ ]:
# 演示多层卷积后尺寸变化
def simulate_conv_layers(input_size, kernel_size, num_layers):
    """模拟多层卷积的尺寸变化"""
    sizes = [input_size]
    current_size = input_size
    
    for i in range(num_layers):
        current_size = current_size - kernel_size + 1
        if current_size <= 0:
            print(f"警告: 在第{i+1}层后,特征图尺寸变为{current_size}!")
            break
        sizes.append(current_size)
    
    return sizes

# 示例
input_size = 28
kernel_size = 5
num_layers = 10

sizes = simulate_conv_layers(input_size, kernel_size, num_layers)

print(f"输入大小: {input_size}×{input_size}")
print(f"卷积核大小: {kernel_size}×{kernel_size}")
print(f"\n各层输出大小:")
for i, size in enumerate(sizes):
    print(f"  第{i}层: {size}×{size}")

# 可视化
plt.figure(figsize=(10, 6))
plt.plot(range(len(sizes)), sizes, marker='o', linewidth=2, markersize=8)
plt.xlabel('层数', fontsize=12)
plt.ylabel('特征图大小', fontsize=12)
plt.title(f'多层卷积后特征图尺寸变化 (核大小={kernel_size}×{kernel_size})', fontsize=14)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\n结论: 不使用填充,特征图会快速缩小!")

### 3.2 填充 (Padding)

在输入周围填充0,以保持输出大小。

填充 $p_h$ 行和 $p_w$ 列后:  
输出大小: $(n_h - k_h + p_h + 1) \times (n_w - k_w + p_w + 1)$

**常用**: 设置 $p_h = k_h - 1$ 和 $p_w = k_w - 1$,使输入输出同大小。

In [ ]:
# 使用PyTorch的Conv2d(带填充)
def test_padding(input_size, kernel_size, padding):
    """测试不同填充的效果"""
    X = torch.rand(1, 1, input_size, input_size)
    conv = nn.Conv2d(1, 1, kernel_size=kernel_size, padding=padding)
    Y = conv(X)
    return Y.shape[2], Y.shape[3]

input_size = 8
kernel_size = 3

print(f"输入大小: {input_size}×{input_size}")
print(f"卷积核大小: {kernel_size}×{kernel_size}\n")

paddings = [0, 1, 2]
for p in paddings:
    h, w = test_padding(input_size, kernel_size, p)
    print(f"填充={p}: 输出大小={h}×{w}")

print("\n观察: 填充=1时,输入输出同大小(3×3卷积核)")

### 3.3 步幅 (Stride)

卷积窗口每次移动的步长。

步幅为 $(s_h, s_w)$ 时:  
输出大小: $\lfloor \frac{n_h - k_h + p_h + s_h}{s_h} \rfloor \times \lfloor \frac{n_w - k_w + p_w + s_w}{s_w} \rfloor$

**作用**: 大步幅可以快速降低特征图尺寸。

In [ ]:
# 测试不同步幅
input_size = 8
kernel_size = 3
padding = 1

strides = [1, 2, 3]

print(f"输入大小: {input_size}×{input_size}")
print(f"卷积核大小: {kernel_size}×{kernel_size}")
print(f"填充: {padding}\n")

results = []
for s in strides:
    X = torch.rand(1, 1, input_size, input_size)
    conv = nn.Conv2d(1, 1, kernel_size=kernel_size, padding=padding, stride=s)
    Y = conv(X)
    h, w = Y.shape[2], Y.shape[3]
    results.append((s, h, w))
    print(f"步幅={s}: 输出大小={h}×{w}")

# 可视化
strides_list = [r[0] for r in results]
output_sizes = [r[1] for r in results]

plt.figure(figsize=(10, 6))
plt.plot(strides_list, output_sizes, marker='o', linewidth=2, markersize=10, color='steelblue')
plt.xlabel('步幅', fontsize=12)
plt.ylabel('输出大小', fontsize=12)
plt.title('步幅对输出大小的影响', fontsize=14)
plt.grid(True, alpha=0.3)
plt.xticks(strides_list)
for s, size in zip(strides_list, output_sizes):
    plt.text(s, size + 0.2, f'{size}×{size}', ha='center', fontsize=10)
plt.tight_layout()
plt.show()

print("\n结论: 增大步幅可快速降低特征图尺寸")

---

## 第四部分: 多通道卷积

### 4.1 多输入通道

彩色图像有RGB三个通道: $3 \times H \times W$

对于 $c_i$ 个输入通道:
- 卷积核形状: $c_i \times k_h \times k_w$
- 对每个通道分别卷积,然后求和
- 输出: 单通道

In [ ]:
def corr2d_multi_in(X, K):
    """多输入通道互相关运算"""
    # 对每个通道分别计算,然后求和
    return sum(corr2d(x, k) for x, k in zip(X, K))

# 示例: 2个输入通道
X = torch.tensor([[[0.0, 1.0, 2.0],
                    [3.0, 4.0, 5.0],
                    [6.0, 7.0, 8.0]],
                   [[1.0, 2.0, 3.0],
                    [4.0, 5.0, 6.0],
                    [7.0, 8.0, 9.0]]])

K = torch.tensor([[[0.0, 1.0],
                    [2.0, 3.0]],
                   [[1.0, 2.0],
                    [3.0, 4.0]]])

print("输入X的形状:", X.shape, "(通道数, 高, 宽)")
print("卷积核K的形状:", K.shape)
print("\n输出:")
Y = corr2d_multi_in(X, K)
print(Y)

print("\n计算过程(第一个输出元素):")
print("通道0: 0×0 + 1×1 + 3×2 + 4×3 =", (0*0 + 1*1 + 3*2 + 4*3).item())
print("通道1: 1×1 + 2×2 + 4×3 + 5×4 =", (1*1 + 2*2 + 4*3 + 5*4).item())
print("总和: 19 + 37 = 56")

### 4.2 多输出通道

为了提取不同特征,通常需要多个输出通道。

$c_o$ 个输出通道需要 $c_o$ 个卷积核:  
卷积核形状: $c_o \times c_i \times k_h \times k_w$

In [ ]:
def corr2d_multi_in_out(X, K):
    """多输入多输出通道互相关运算"""
    return torch.stack([corr2d_multi_in(X, k) for k in K], 0)

# 构造3个输出通道的卷积核
K_multi = torch.stack((K, K + 1, K + 2), 0)

print("卷积核形状:", K_multi.shape, "(输出通道, 输入通道, 高, 宽)")
print("\n输出:")
Y_multi = corr2d_multi_in_out(X, K_multi)
print("输出形状:", Y_multi.shape, "(输出通道, 高, 宽)")
print(Y_multi)

### 4.3 1×1卷积

$1 \times 1$ 卷积不改变空间维度,只改变通道数。

**作用**:
- 改变通道数(升维或降维)
- 增加非线性
- 相当于在每个像素位置应用全连接层

In [ ]:
# 1×1卷积示例
X = torch.randn(1, 3, 8, 8)  # batch=1, channels=3, H=8, W=8

# 3个输入通道 → 10个输出通道
conv1x1 = nn.Conv2d(3, 10, kernel_size=1)

Y = conv1x1(X)

print(f"输入形状: {X.shape}")
print(f"输出形状: {Y.shape}")
print(f"\n参数量: {sum(p.numel() for p in conv1x1.parameters())}")
print(f"  权重: 3×10×1×1 = 30")
print(f"  偏置: 10")
print(f"  总计: 40")

print("\n观察: 空间维度(8×8)不变,通道数从3变为10")

---

## 第五部分: 池化层

### 5.1 池化的作用

**目的**:
1. 降低特征图分辨率
2. 减少计算量
3. 增加平移不变性
4. 扩大感受野

**特点**: 没有可学习的参数

### 5.2 最大池化和平均池化

In [ ]:
def pool2d(X, pool_size, mode='max'):
    """池化运算"""
    p_h, p_w = pool_size
    Y = torch.zeros((X.shape[0] - p_h + 1, X.shape[1] - p_w + 1))
    for i in range(Y.shape[0]):
        for j in range(Y.shape[1]):
            if mode == 'max':
                Y[i, j] = X[i:i + p_h, j:j + p_w].max()
            elif mode == 'avg':
                Y[i, j] = X[i:i + p_h, j:j + p_w].mean()
    return Y

# 测试
X = torch.tensor([[0.0, 1.0, 2.0],
                   [3.0, 4.0, 5.0],
                   [6.0, 7.0, 8.0]])

print("输入:")
print(X)

print("\n2×2最大池化:")
Y_max = pool2d(X, (2, 2), 'max')
print(Y_max)

print("\n2×2平均池化:")
Y_avg = pool2d(X, (2, 2), 'avg')
print(Y_avg)

# 可视化
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].imshow(X, cmap='viridis')
axes[0].set_title('输入 (3×3)', fontsize=12)
axes[0].axis('off')

axes[1].imshow(Y_max, cmap='viridis')
axes[1].set_title('最大池化 (2×2)', fontsize=12)
axes[1].axis('off')

axes[2].imshow(Y_avg, cmap='viridis')
axes[2].set_title('平均池化 (2×2)', fontsize=12)
axes[2].axis('off')

plt.tight_layout()
plt.show()

### 5.3 池化的填充和步幅

In [ ]:
# PyTorch的池化层
X = torch.arange(16, dtype=torch.float32).reshape(1, 1, 4, 4)

print("输入 (4×4):")
print(X.squeeze())

# 3×3池化窗口,步幅=3(默认)
pool2d = nn.MaxPool2d(3)
print("\n3×3最大池化(步幅=3):")
print(pool2d(X).squeeze())

# 3×3池化窗口,步幅=2,填充=1
pool2d = nn.MaxPool2d(3, stride=2, padding=1)
print("\n3×3最大池化(步幅=2, 填充=1):")
print(pool2d(X).squeeze())

# 不同形状的池化窗口
pool2d = nn.MaxPool2d((2, 3), stride=(2, 3), padding=(0, 1))
print("\n2×3最大池化(步幅=(2,3), 填充=(0,1)):")
print(pool2d(X).squeeze())

### 5.4 池化的平移不变性

In [ ]:
# 演示池化的平移不变性
X1 = torch.tensor([[0., 1., 2.],
                    [3., 4., 5.],
                    [6., 7., 8.]])

# 向右平移1像素
X2 = torch.tensor([[0., 0., 1.],
                    [0., 3., 4.],
                    [0., 6., 7.]])

pool = nn.MaxPool2d(2, stride=1)

Y1 = pool(X1.unsqueeze(0).unsqueeze(0)).squeeze()
Y2 = pool(X2.unsqueeze(0).unsqueeze(0)).squeeze()

print("原始输入:")
print(X1)
print("\n池化输出:")
print(Y1)

print("\n平移后输入:")
print(X2)
print("\n池化输出:")
print(Y2)

print("\n观察: 即使输入平移,池化后的输出仍能保持相似的模式")

---

## 小结

### 核心概念

1. **卷积神经网络的优势**:
   - 参数共享: 大幅减少参数量
   - 局部连接: 提取局部特征
   - 平移不变性: 位置无关的特征提取

2. **卷积运算**:
   - 互相关: $[\mathbf{Y}]_{i,j} = \sum_a \sum_b [\mathbf{K}]_{a,b} \cdot [\mathbf{X}]_{i+a, j+b}$
   - 输出大小: $(n_h - k_h + 1) \times (n_w - k_w + 1)$

3. **填充和步幅**:
   - 填充: 保持输出大小,设置 $p = k - 1$
   - 步幅: 降低输出大小,加速计算

4. **多通道**:
   - 输入通道: 彩色图像(RGB)
   - 输出通道: 提取不同特征
   - 1×1卷积: 改变通道数

5. **池化层**:
   - 最大池化: 取窗口最大值
   - 平均池化: 取窗口平均值
   - 作用: 降采样,增加不变性

### 关键公式

**输出大小**(考虑填充和步幅):

$$\text{output\_size} = \left\lfloor \frac{n + 2p - k}{s} \right\rfloor + 1$$

其中:
- $n$: 输入大小
- $k$: 卷积核大小
- $p$: 填充
- $s$: 步幅

### 实践要点

1. **卷积核大小**: 常用3×3, 5×5, 7×7(奇数)
2. **填充**: same填充使输入输出同大小
3. **步幅**: 步幅>1用于降采样
4. **池化**: 2×2最大池化最常用
5. **通道数**: 逐层增加(64→128→256→512)

## 练习

1. **边缘检测**: 设计水平和45°边缘检测核
2. **输出计算**: 给定输入28×28,核5×5,填充2,步幅1,计算输出大小
3. **参数量**: 计算一个卷积层的参数量(输入3通道,输出64通道,核3×3)
4. **感受野**: 计算3层3×3卷积后的感受野大小
5. **池化对比**: 在真实图像上对比最大池化和平均池化的效果